# Prioritätswarteschlange

## Agenda

1. „Prioritätswarteschlange“
2. Naive Implementierung
3. Heap
    - Funktionsweise
    - Implementierung
    - Laufzeitanalyse
4. Heap-Konstruktion
5. Heapsort

## 1. Prioritätswarteschlange

Der Prioritätswarteschlangen-ADT ähnelt einer Warteschlange, da Werte konzeptionell an einem Ende hinzugefügt und am anderen Ende entnommen werden. Werte werden jedoch nicht in FIFO-Reihenfolge aus einer Prioritätswarteschlange entfernt. Stattdessen hat jeder Wert in einer Prioritätswarteschlange eine implizite „Priorität“, und der *Wert mit der höchsten Priorität wird immer zuerst entfernt*, unabhängig davon, wann er eingefügt wurde.

Als Beispiel kann man sich die wartenden Patienten in einer Notaufnahme vorstellen. Die Reihenfolge der Behandlung folgt nicht dem "First-Come-First-Serve"-Prinzip, sondern die Patienten werden gemäß medizinischer Notwendigkeit behandelt. So dass Schwerverletzte schneller behandelt werden, als Leichtverletzte.

## 2. Naive Implementierung
Verwendung eines sortierten Python-Arrays, bei dem das Element mit der höchsten Priorität am Ende des Arrays steht. Dabei verwenden wir den Wert des Elements auch als Priorität.

In [ ]:
class PriorityQueue:
    def __init__(self):
        self.data = []
        
    def add(self, x): # O(N)
        # insert x in the correct place (to keep data in ascending order)
        for i in range(len(self.data)):
            if self.data[i] > x:
                self.data.insert(i, x)
                break
        else:
            self.data.append(x)
    
    def max(self): # O(1)
        assert len(self) > 0
        return self.data[-1]
        
    def pop_max(self): # O(1)
        assert len(self) > 0
        ret = self.data[-1]
        del self.data[-1]
        return ret
    
    def __bool__(self):
        return len(self.data) > 0

    def __len__(self):
        return len(self.data)

    def __repr__(self):
        return repr(self.data)

In [ ]:
import random

pq = PriorityQueue()

vals = random.sample(range(100), 10)
for x in vals:
    pq.add(x)

In [ ]:
pq

In [ ]:
while pq:
    pq.pop_max()
    print(pq)

## 3. Heap

Ein Heap ist eine Implementierung einer Prioritätswarteschlange, die eine *partielle Ordnung* auf ihren Inhalten erzwingt. Ein Heap nimmt die Form eines *vollständigen binären Baums* an, bei dem jeder Knoten die *Heap-Eigenschaft* erfüllt, d.h., dass der Wert in einem gegebenen Knoten der maximale Wert im Teilbaum ist, dessen Wurzel er ist.

<div style="border: 2px solid #1a73e8; border-radius: 8px; padding: 12px; background: #e8f0fe;">
<b>Heap-Eigenschaft:</b><br>
In einem <b>Max-Heap</b> ist jeder Elternknoten größer oder gleich seinen beiden Kindknoten.<br>
In einem <b>Min-Heap</b> ist jeder Elternknoten kleiner oder gleich seinen beiden Kindknoten.
</div>

### Mechanik

Die Heap-Eigenschaft wird bei Einfügungen und Löschungen durch die Algorithmen „Bubble Up“ und "Trickle Down" aufrechterhalten.
> **Bubble up (heapify up) im Heap:**  
> Wird beim Einfügen eines neuen Elements benötigt.  
> Das neue Element wird von unten nach oben mit seinen Eltern verglichen und ggf. vertauscht, bis die Heap-Eigenschaft wiederhergestellt ist.

![](images/heap_insertion_bubble_up.png)


> **Trickledown (heapify down) im Heap:**  
> Wird immer dann benötigt, wenn das Wurzel-Element entfernt oder ersetzt wurde.  
> Das neue Element an der Wurzel wird so lange mit seinen Kindern verglichen und ggf. vertauscht, bis die Heap-Eigenschaft wiederhergestellt ist.

![](images/heap_deletion_trickledown.png)

Beachte, dass der „Trickle Down“-Algorithmus auch als eine Methode zum „Re-Heapifizieren“ eines Baums betrachtet werden kann, bei dem alle Knoten außer der Wurzel die Heap-Eigenschaft erfüllen.

### Implementierung

In [ ]:
class Heap:
    def __init__(self):
        self.data = []
        
    @staticmethod
    def _parent(idx):
        return (idx - 1) // 2
    
    @staticmethod
    def _left(idx):
        return idx*2 + 1

    @staticmethod
    def _right(idx):
        return idx*2 + 2

    def add(self, x):
        self.data.append(x) # add the value at the bottom right of the tree
        
        # carry out the bubble-up algorithm
        idx = len(self.data) - 1
        while idx > 0:
            pidx = Heap._parent(idx)
            if self.data[pidx] < self.data[idx]: # i.e., max-heap property is false
                self.data[pidx], self.data[idx] = self.data[idx], self.data[pidx]
                idx = pidx
            else:
                break    
                
    def max(self):
        assert len(self) > 0
        return self.data[0]


    def pop_max(self):
        assert len(self) > 0
        ret = self.data[0]
        
        # move the bottom-right value to the root
        self.data[0] = self.data[-1]
        del self.data[-1]
        
        # re-heapify using the trickle down algorithm
        idx = 0
        while idx < len(self.data):
            lidx = Heap._left(idx)
            ridx = Heap._right(idx)
            maxidx = idx
            if lidx < len(self.data) and self.data[lidx] > self.data[idx]:
                maxidx = lidx
            if ridx < len(self.data) and self.data[ridx] > self.data[maxidx]:
                maxidx = ridx
            if maxidx != idx:
                self.data[idx], self.data[maxidx] = self.data[maxidx], self.data[idx]
                idx = maxidx
            else:
                break
        
        return ret
            

    def __bool__(self):
        return len(self.data) > 0

    def __len__(self):
        return len(self.data)

    def __repr__(self):
        return repr(self.data)

In [ ]:
import random

h = Heap()

#vals = random.sample(range(100), 10)
vals = range(10)
for x in vals:
    h.add(x)
    print(h)

In [ ]:
h

In [ ]:
while h:
    print(h.pop_max())

Pretty Printing eines Heaps ist keine einfache Aufgabe. Hier ist ein schöner Algorithmus zum Pretty Printing eines Heaps (entnommen von https://gist.github.com/ydm/4f0c948bc0d151631621):

In [ ]:
from math import log
first = lambda h: 2**h - 1      # H stands for level height
last = lambda h: first(h + 1)
level = lambda heap, h: heap[first(h):last(h)]
prepare = lambda e, field: str(e).center(field)

def heap_print(heap, width=None):
    if width is None:
        width = max(len(str(e)) for e in heap.data)
    height = int(log(len(heap), 2)) + 1
    gap = ' ' * width
    for h in range(height):
        below = 2 ** (height - h - 1)
        field = (2 * below - 1) * width
        print(gap.join(prepare(e, field) for e in level(heap.data, h)))

In [ ]:
heap_print(h)

### Laufzeitanalyse

Die Laufzeitanalyse hängt davon ab, wie oft wir hoch "bubbeln" oder runter "trickeln" müssen. Also davon welche Höhe (Anzahl Ebenen) ein Baum hat. 

Für die Höhe in einem vollständigen binären Baum gilt:

![](images/heap_height_logarithmic.png)

Das heißt, beim Hinzufügen oder Entfernen von Elementen aus einem Heap müssen wir höchstens $O(h)$ Operationen durchführen, wobei $h$ die Anzahl der Ebenen im Heap ist, um sicherzustellen, dass die Heap-Eigenschaft überall eingehalten wird. Da der Heap die Form eines vollständigen binären Baums hat und die Höhe $h$ des Baums $O(\log N)$ ist, wobei $N$ die Anzahl der Elemente im Baum ist, schließen wir, dass **die Hinzufüge- und Entferne-Operationen im Heap beide $O(\log N)$ sind**.

## 4. Heap-Konstruktion

Wenn wir einen Heap aus $N$ Elementen konstruieren, indem wir einfach `add` $N$-mal aufrufen, ist leicht zu erkennen, dass dieser Ansatz eine Laufzeitkomplexität von $O(N \log N)$ hat.

Können wir es besser machen?

Ja! Wenn uns eine Liste von $N$ Werten gegeben ist, aus der wir einen Heap konstruieren sollen, beginnen wir mit der Beobachtung, dass wir die Liste als Darstellung eines vollständigen binären Baums interpretieren können. In diesem Baum sind die einzigen Werte, die die Heap-Eigenschaft verletzen können, diejenigen, die sich in *inneren Knoten* befinden (d.h. Knoten mit mindestens einem Kind).

Erinneren wir uns daran, dass wenn uns ein vollständiger Baum gegeben ist, bei dem nur der Wurzelknoten die Heap-Eigenschaft verletzt, wir ihn durch Anwendung des Trickle-Down-Algorithmus beginnend an der Wurzel wieder heapifizieren können.

Daher müssen wir, um aus einer Liste einen Heap zu bauen, den Trickle-Down-Algorithmus nur auf jeden Knoten anwenden, beginnend beim tiefsten, ganz rechts liegenden inneren Knoten, und dann nach oben bis zur Wurzel.

![](images/heap_innernodes_leaves.png)

![](/images/heap_innernodes_leaves.png)

In einem Baum mit insgesamt $N$ Knoten gibt es nur $\lfloor \frac{N-1}{2} \rfloor$ innere Knoten, was bedeutet, dass wir den Trickle-Down-Algorithmus nur auf ungefähr *die Hälfte* der Knoten im Baum anwenden müssen. Beachte außerdem, dass die Laufzeitkomplexität von Trickle-Down von der Höhe des Knotens abhängt, welche nur an der Wurzel mit der Höhe des Baumes übereinstimmt. Wir werden dies nicht beweisen, aber der Aufbau eines Heaps mit diesem Ansatz ergibt einen Algorithmus mit einer Laufzeitkomplexität von $O(N)$.

In [ ]:
class Heap(Heap):
    def __init__(self, iterable=None):
        if not iterable:
            self.data = []
        else:
            self.data = list(iterable)
            last_internal_idx = Heap._parent(len(self.data)-1)
            for i in range(last_internal_idx, -1, -1):
                self._heapify(i)
        
    def _heapify(self, idx):
        # re-heapify using the trickle down algorithm, starting at idx
        while idx < len(self.data):
            lidx = Heap._left(idx)
            ridx = Heap._right(idx)
            maxidx = idx
            if lidx < len(self.data) and self.data[lidx] > self.data[idx]:
                maxidx = lidx
            if ridx < len(self.data) and self.data[ridx] > self.data[maxidx]:
                maxidx = ridx
            if maxidx != idx:
                self.data[idx], self.data[maxidx] = self.data[maxidx], self.data[idx]
                idx = maxidx
            else:
                break

    def pop_max(self):
        assert len(self) > 0
        ret = self.data[0]
        
        # move the bottom-right value to the root
        self.data[0] = self.data[-1]
        del self.data[-1]
        
        self._heapify(0)
        
        return ret

In [ ]:
import random

h = Heap()

vals = random.sample(range(100), 10)
for x in vals:
    h.add(x)

In [ ]:
h

In [ ]:
while h:
    print(h.pop_max())

In [ ]:
h = Heap(random.sample(range(100), 10)) # use the new heap initializer

while h:
    print(h.pop_max())

## 5. Heapsort

Wir können einen Heap verwenden, um einen effizienten Sortieralgorithmus zu implementieren: Heapsort!

In [ ]:
def heapsort(iterable):
    h = Heap(iterable) # O(N) -- build a heap using approach described above
    ret = []
    while h: # O(N log N)
        ret.append(h.pop_max())
    ret.reverse() # reverse so that we have ascending order
    return ret

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

lst = list(range(100))
random.shuffle(lst)
plt.plot(lst, 'ro');

In [ ]:
plt.plot(heapsort(lst), 'ro');

In [ ]:
def insertion_sort(lst):
    for i in range(1, len(lst)):
        for j in range(i, 0, -1):
            if lst[j-1] > lst[j]:
                lst[j-1], lst[j] = lst[j], lst[j-1] # swap
            else:
                break

In [ ]:
import timeit

def time_insertionsort(n):
    return timeit.timeit('insertion_sort(lst)',
                         f'lst = random.sample(range(1_000_000), {n})',
                         globals=globals(),
                         number=1)

def time_heapsort(n):
    return timeit.timeit('heapsort(lst)',
                         f'lst = random.sample(range(1_000_000), {n})',
                         globals=globals(),
                         number=1)

In [ ]:
ns = np.linspace(100, 2000, 50, dtype=np.int_)
plt.plot(ns, [time_insertionsort(n) for n in ns], 'ro')
plt.plot(ns, [time_heapsort(n) for n in ns], 'b^');

In [ ]:
ns = np.linspace(100, 10000, 50, dtype=np.int_)
plt.plot(ns, [time_heapsort(n) for n in ns], 'b^');

Heapsort ist der erste Sortieralgorithmus, den wir mit $O(N \log N)$ kennengelernt haben.